# Step 7 — ECAPA + CM under noise (AASIST α=0.30, optional LFCC)

After Wave-U-Net gating failed, pivot to the real SASV stack under the same SNR grid.

| ID | Score |
|----|-------|
| **B0** | ECAPA cosine only |
| **B1_w030** | `0.30·s_asv + 0.70·s_cm` (AASIST P_bona) — locked α |
| **B1_sum** | `s_asv + s_cm` (AASIST) |
| **B2_lfcc** | `s_asv + s_cm` (app LFCC) — optional |

- Enrol: **clean** ECAPA speaker models  
- Test: MUSAN noise at **clean / 15 / 10 / 5 / 0** (same recipe as notebook 03)  
- CM sees the **same noisy** waveform as ECAPA  

`SMOKE = True` → 1500 trials (debug). `False` → full dev (~29.5k).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "noise_gated_lib.py").exists():
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "sasv_la2019"))

from noise_gated_lib import (
    DEFAULT_ALPHA_AASIST,
    DEFAULT_LA,
    DEFAULT_SASV,
    DEFAULT_SNRS_DB,
    RUNS_DIR,
    build_speaker_models,
    configure_app_la_cm,
    cosine,
    eers_from_preds,
    embed_waveform,
    ensure_dirs,
    ensure_sasv_on_path,
    ensure_server_on_path,
    load_aasist_cm,
    load_app_ecapa,
    load_waveform,
    maybe_noise_waveform,
    patch_speechbrain_windows_lazy_import,
    read_trials,
    resolve_audio_path,
    resolve_noise_bank,
    save_json,
    score_aasist_waveform,
    score_app_la_waveform,
    snr_tag,
    trial_key_counts,
    weighted_asv_cm,
    write_score_csv,
)

ensure_dirs()
ensure_server_on_path()
patch_speechbrain_windows_lazy_import()

SMOKE = True
SPLIT = "dev"
MAX_TRIALS = 1500 if SMOKE else 0
ALPHA = DEFAULT_ALPHA_AASIST  # 0.30
RUN_LFCC = True  # set False to skip B2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 20260926

NOISE_ROOT = Path(r"D:/downloads/musan/musan/noise")
noise_bank, noise_used = resolve_noise_bank(NOISE_ROOT)

OUT = RUNS_DIR / "step7_cm_under_noise"
OUT.mkdir(parents=True, exist_ok=True)

print("device", DEVICE, "split", SPLIT, "max_trials", MAX_TRIALS, "alpha", ALPHA)
print("SNR grid", [snr_tag(s) for s in DEFAULT_SNRS_DB])
print("MUSAN", noise_used, "clips", len(noise_bank))
print("RUN_LFCC", RUN_LFCC, "out", OUT)


### Load ECAPA + AASIST (+ optional LFCC) and enrol


In [ ]:
sasv = ensure_sasv_on_path(DEFAULT_SASV)
trials = read_trials(sasv, SPLIT, max_trials=MAX_TRIALS)
print(trial_key_counts(trials))

classifier = load_app_ecapa(device=DEVICE)
aasist_model, _aasist_cfg = load_aasist_cm(device=DEVICE)
print("AASIST ready")

if RUN_LFCC:
    configure_app_la_cm("lfcc")
    print("LFCC CM configured")

spk_models = build_speaker_models(classifier, DEFAULT_LA, SPLIT, DEVICE)
print("speakers", len(spk_models))


### Score all SNRs

Caches embeddings / CM scores per `(test_utt, snr)` so repeated trials reuse work.


In [ ]:
def utt_seed(test_utt: str, snr_db) -> int:
    return int(SEED + (hash((test_utt, snr_db)) % (2**31)))


@torch.inference_mode()
def score_snr(trials, snr_db):
    tag = snr_tag(snr_db)
    emb_cache: dict[str, np.ndarray] = {}
    aasist_cache: dict[str, tuple[float, float]] = {}
    lfcc_cache: dict[str, tuple[float, float]] = {}
    noisy_cache: dict[str, torch.Tensor] = {}

    systems = ["B0", "B1_w030", "B1_sum"] + (["B2_lfcc"] if RUN_LFCC else [])
    preds = {s: [] for s in systems}
    rows = {s: [] for s in systems}
    keys = []

    for trial in tqdm(trials, desc=f"cm:{tag}"):
        utt = trial.test_utt
        if utt not in noisy_cache:
            path = resolve_audio_path(DEFAULT_LA, SPLIT, utt)
            clean = load_waveform(path)
            rng = np.random.default_rng(utt_seed(utt, snr_db))
            noisy_cache[utt] = maybe_noise_waveform(
                clean, snr_db=snr_db, noise_bank=noise_bank, rng=rng
            )
        noisy = noisy_cache[utt]

        if utt not in emb_cache:
            emb_cache[utt] = embed_waveform(classifier, noisy, DEVICE)
        if utt not in aasist_cache:
            aasist_cache[utt] = score_aasist_waveform(aasist_model, noisy, DEVICE)
        if RUN_LFCC and utt not in lfcc_cache:
            lfcc_cache[utt] = score_app_la_waveform(noisy, DEVICE)

        s_asv = float(cosine(spk_models[trial.speaker_id], emb_cache[utt]))
        s_cm_a, p_spf_a = aasist_cache[utt]
        s_b0 = s_asv
        s_w = weighted_asv_cm(s_asv, s_cm_a, ALPHA)
        s_sum = float(s_asv + s_cm_a)

        pack = {
            "B0": (s_b0, None, None),
            "B1_w030": (s_w, s_cm_a, p_spf_a),
            "B1_sum": (s_sum, s_cm_a, p_spf_a),
        }
        if RUN_LFCC:
            s_cm_l, p_spf_l = lfcc_cache[utt]
            pack["B2_lfcc"] = (float(s_asv + s_cm_l), s_cm_l, p_spf_l)

        for name, (score, s_cm, p_spf) in pack.items():
            preds[name].append(score)
            row = {
                "speaker_id": trial.speaker_id,
                "test_utt": utt,
                "key": trial.key,
                "snr": tag,
                "s_asv": s_asv,
                "score": score,
            }
            if s_cm is not None:
                row["s_cm"] = s_cm
                row["p_spoof"] = p_spf
            if name == "B1_w030":
                row["alpha"] = ALPHA
            rows[name].append(row)

        keys.append(trial.key)

    return rows, preds, keys, systems


all_summaries = {}
for snr_db in DEFAULT_SNRS_DB:
    tag = snr_tag(snr_db)
    rows, preds, keys, systems = score_snr(trials, snr_db)
    out = OUT / f"{SPLIT}_{tag}"
    out.mkdir(parents=True, exist_ok=True)

    summaries = {}
    for name in systems:
        fields = ["speaker_id", "test_utt", "key", "snr", "s_asv", "score"]
        if name != "B0":
            fields += ["s_cm", "p_spoof"]
        if name == "B1_w030":
            fields.append("alpha")
        write_score_csv(out / f"{name}_scores.csv", rows[name], fields)
        summaries[name] = eers_from_preds(preds[name], keys)
        summaries[name]["n_trials"] = len(keys)

    save_json(out / "metrics.json", summaries)
    all_summaries[tag] = summaries
    print(tag, {k: round(v["sasv_eer_percent"], 3) for k, v in summaries.items()})

save_json(
    OUT / f"{SPLIT}_all_snrs.json",
    {
        "split": SPLIT,
        "max_trials": MAX_TRIALS,
        "smoke": SMOKE,
        "alpha": ALPHA,
        "run_lfcc": RUN_LFCC,
        "device": DEVICE,
        "summaries": all_summaries,
    },
)
print("wrote", OUT / f"{SPLIT}_all_snrs.json")


### Quick table


In [ ]:
import pandas as pd

rows = []
for snr, systems in all_summaries.items():
    for name, m in systems.items():
        rows.append(
            {
                "snr": snr,
                "system": name,
                "sasv_eer_percent": m["sasv_eer_percent"],
                "sv_eer_percent": m["sv_eer_percent"],
                "spf_eer_percent": m["spf_eer_percent"],
            }
        )
df = pd.DataFrame(rows)
pivot = df.pivot_table(index="snr", columns="system", values="sasv_eer_percent")
order = [snr_tag(s) for s in DEFAULT_SNRS_DB]
pivot = pivot.reindex(order)
display(pivot.round(3))
pivot.to_csv(OUT / f"matrix_sasv_eer_{SPLIT}.csv")
df.to_csv(OUT / f"metrics_long_{SPLIT}.csv", index=False)
print("Next → 08_cm_noise_matrix_and_claim.ipynb")
